# 🔀 Project 7.4 — Merge K Sensor Logs & Top-K Fault Analysis
**DSA for Mechatronics · Week 07 — Heaps & Priority Queues**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq, math
from collections import Counter, defaultdict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
K sensor nodes each produce a **sorted** log of (timestamp, fault_code) events.
The control room must:
1. **Merge** all K logs into one chronological stream — heap gives O(n log K)
2. Find the **top-K most frequent fault codes** across the merged log
3. Run **Dijkstra's shortest path** on a weighted sensor communication graph
   to find the fastest route for a maintenance alert


## Step 1 — Generate K sorted sensor logs

In [ ]:
# ── Personalised parameters ───────────────────────────────────────
K_LOGS       = random.randint(4, 7)
LOG_LEN_MIN  = random.randint(8, 12)
LOG_LEN_MAX  = random.randint(16, 24)
TS_RANGE     = (0, random.randint(1200, 2400))   # ms
K_FREQUENT   = random.randint(3, 5)              # top-K faults to report

FAULT_CODES  = [f"F{i:03d}" for i in range(10, 40)]

logs = []
for k in range(K_LOGS):
    n   = random.randint(LOG_LEN_MIN, LOG_LEN_MAX)
    ts  = sorted(random.sample(range(*TS_RANGE), n))
    log = [(t, random.choice(FAULT_CODES)) for t in ts]
    logs.append(log)

total_events = sum(len(l) for l in logs)
print(f"Sensor log parameters:")
print(f"  Nodes (K)      : {K_LOGS}")
print(f"  Events/node    : {LOG_LEN_MIN} – {LOG_LEN_MAX}")
print(f"  Total events   : {total_events}")
print(f"  Timestamp range: {TS_RANGE[0]} – {TS_RANGE[1]} ms")
print(f"  Top-K faults   : {K_FREQUENT}")
print()
for k, log in enumerate(logs):
    preview = ", ".join(f"({t}ms,{f})" for t,f in log[:4])
    print(f"  Node {k+1}: {len(log)} events — {preview}...")


## Step 2 — Merge K sorted logs with a min-heap (O(n log K))

In [ ]:
def merge_k_sorted_logs(logs):
    """
    Merge K sorted lists using a min-heap.

    Algorithm:
      1. Push (timestamp, fault_code, log_index, pos_in_log) for the first
         element of each non-empty log.
      2. While heap not empty:
         a. Pop the smallest (timestamp, ...) — this is the next event globally.
         b. If that log has a next element, push it.

    Time:  O(n log K)  where n = total events, K = number of logs.
    Space: O(K)        heap never exceeds K entries.
    """
    heap = []
    # seed heap with first element of each log
    for i, log in enumerate(logs):
        if log:
            ts, fc = log[0]
            heapq.heappush(heap, (ts, fc, i, 0))

    merged = []
    while heap:
        ts, fc, log_idx, pos = heapq.heappop(heap)
        merged.append((ts, fc, log_idx + 1))   # 1-indexed node
        next_pos = pos + 1
        if next_pos < len(logs[log_idx]):
            nts, nfc = logs[log_idx][next_pos]
            heapq.heappush(heap, (nts, nfc, log_idx, next_pos))

    return merged

merged = merge_k_sorted_logs(logs)

# Verify chronological order
is_sorted_ok = all(merged[i][0] <= merged[i+1][0] for i in range(len(merged)-1))

print(f"Merged log ({len(merged)} events, sorted: {is_sorted_ok}):")
print(f"  {'#':>4}  {'Time':>8}  {'Fault':<6}  Node")
print(f"  {'─'*4}  {'─'*8}  {'─'*6}  {'─'*6}")
for i, (ts, fc, node) in enumerate(merged[:16]):
    print(f"  {i:4d}  {ts:6d}ms  {fc:<6}  N{node}")
if len(merged) > 16:
    print(f"  ... and {len(merged)-16} more events")
print(f"\n  Chronologically sorted: {'✅ YES' if is_sorted_ok else '❌ NO'}")


## Step 3 — Top-K most frequent fault codes (size-K min-heap)

In [ ]:
def top_k_frequent(events, k):
    """
    Find top-K most frequent fault codes.

    Algorithm:
      1. Count all fault code frequencies.
      2. Maintain a min-heap of size k containing (count, fault_code).
         - For each (count, code): if heap has < k items → push.
           Else if count > heap[0].count → heapreplace (evict least frequent).
      3. Result: the k items remaining in the heap.

    Time:  O(n + f log k)  where f = number of distinct fault codes.
    Space: O(f + k).
    """
    freq = Counter(fc for _, fc, _ in events)
    heap = []
    for fc, cnt in freq.items():
        if len(heap) < k:
            heapq.heappush(heap, (cnt, fc))
        elif cnt > heap[0][0]:
            heapq.heapreplace(heap, (cnt, fc))
    # sort descending by count for display
    return sorted(heap, key=lambda x: -x[0])

top_faults = top_k_frequent(merged, K_FREQUENT)
all_freq   = Counter(fc for _, fc, _ in merged)

print(f"Top {K_FREQUENT} most frequent fault codes:")
print(f"  {'Rank':>5}  {'Code':<6}  {'Count':>7}  {'% of total':>12}  Bar")
print(f"  {'─'*5}  {'─'*6}  {'─'*7}  {'─'*12}  {'─'*20}")
for rank, (cnt, fc) in enumerate(top_faults, 1):
    pct = cnt / len(merged) * 100
    bar = "█" * cnt
    print(f"  {rank:5d}  {fc:<6}  {cnt:7d}  {pct:11.1f}%  {bar}")

print(f"\n  Distinct fault codes : {len(all_freq)}")
print(f"  Total events         : {len(merged)}")


## Step 4 — Dijkstra's shortest path on sensor communication graph

In [ ]:
def dijkstra(graph, src):
    """
    Dijkstra's shortest path from src to all nodes.

    Uses a min-heap of (distance, node).
    O((V + E) log V) with heap.
    """
    dist = {node: float('inf') for node in graph}
    dist[src] = 0
    heap = [(0, src)]
    prev = {node: None for node in graph}

    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]: continue   # stale entry
        for v, w in graph[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(heap, (nd, v))
    return dist, prev

def reconstruct_path(prev, src, dst):
    path = []
    cur = dst
    while cur is not None:
        path.append(cur)
        cur = prev[cur]
    path.reverse()
    return path if path[0] == src else []

# Build random weighted sensor communication graph
N_NODES = K_LOGS + random.randint(2, 4)
node_ids = [f"N{i+1}" for i in range(N_NODES)]
graph = defaultdict(list)
# Connect as a random connected graph
random.shuffle(node_ids)
for i in range(1, N_NODES):
    u = node_ids[random.randint(0, i-1)]
    v = node_ids[i]
    w = random.randint(5, 50)
    graph[u].append((v, w))
    graph[v].append((u, w))
# add a few extra edges
for _ in range(N_NODES):
    u, v = random.sample(node_ids, 2)
    w = random.randint(5, 50)
    graph[u].append((v, w))
    graph[v].append((u, w))
for n in node_ids:
    if n not in graph:
        graph[n] = []

src_node = node_ids[0]
dst_node = node_ids[-1]
dist, prev = dijkstra(graph, src_node)
shortest   = dist[dst_node]
path       = reconstruct_path(prev, src_node, dst_node)

print(f"Dijkstra on sensor network ({N_NODES} nodes):")
print(f"  Source      : {src_node}")
print(f"  Destination : {dst_node}")
print(f"  Shortest dist: {shortest} ms latency")
print(f"  Path        : {' → '.join(path)}")
print()
print(f"  All shortest distances from {src_node}:")
print(f"  {'Node':>6}  {'Distance':>10}")
print(f"  {'─'*6}  {'─'*10}")
for n in sorted(node_ids):
    d = dist[n]
    ds = f"{d}" if d != float('inf') else "∞"
    print(f"  {n:>6}  {ds:>10}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " MERGE K LOGS & TOP-K FAULT ANALYSIS — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  LOG PARAMETERS" + " "*(W-16) + "║")
print(f"║  {'Sensor nodes (K)':<28}: {K_LOGS:<26}║")
print(f"║  {'Total events':<28}: {total_events:<26}║")
print(f"║  {'Chronological sort':<28}: {'YES ✅' if is_sorted_ok else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  TOP-K FAULT CODES" + " "*(W-19) + "║")
for rank, (cnt, fc) in enumerate(top_faults, 1):
    pct = cnt / len(merged) * 100
    print(f"║  {'#'+str(rank)+' '+fc:<28}: {cnt} events ({pct:.1f}%){'':<12}║")
print(f"║  {'Distinct fault codes':<28}: {len(all_freq):<26}║")
print("╠" + "═"*W + "╣")
print("║  DIJKSTRA SHORTEST PATH" + " "*(W-24) + "║")
print(f"║  {'Network nodes':<28}: {N_NODES:<26}║")
print(f"║  {'Source → Destination':<28}: {src_node} → {dst_node}{'':<20}║")
print(f"║  {'Shortest path length':<28}: {shortest} ms{'':<22}║")
print(f"║  {'Path':<28}: {' → '.join(path)}{'':<10}║"[:W+2]+"║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which heap concept did you find most important, and why?**
Pick one technique (sift-up, sift-down, two-heap median, heapify…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
